# Time Series Analysis by Primary Diagnosis (AutoML)

Time series by primary diagnosis: selection of viable series by temporal density, **AutoML** (FLAML) modeling to choose the best model per series, and export of summaries, charts, and evaluation metrics.

## 1. Import data

**Summary:** Loads the classified Excel, filters to the period 2022-01-01 to 2022-12-31 (365 days), and aggregates counts by **day** and **primary_diagnosis** (disease). Lacunas (dias sem casos) serão preenchidas com média móvel nas etapas seguintes. Output is a daily table (date, doenca, casos) used in the next steps.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import zipfile
import pandas as pd
import numpy as np
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find repo root from current working directory and parents."""
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from cwd={start}. Open notebook inside repo workspace."
    )


ROOT = find_project_root(Path.cwd().resolve())

PATH_INPUT = ROOT / "data" / "results" / "agent_outputs" / "agent_outputs_dataset_sintomas_grupos_classificado.xlsx"
PATH_OUTPUT = ROOT / "data" / "results" / "time_series_outputs"
PATH_SUMMARIES = PATH_OUTPUT / "summaries"
PATH_CHARTS = PATH_OUTPUT / "charts"
os.makedirs(PATH_OUTPUT, exist_ok=True)
os.makedirs(PATH_SUMMARIES, exist_ok=True)
os.makedirs(PATH_CHARTS, exist_ok=True)

DATE_START = "2022-01-01"
DATE_END = "2022-12-31"  # 365 dias no ano
TRAIN_DAYS = 252   # treino = primeiros 252 dias
TEST_DAYS = 113    # teste = últimos 113 dias (252 + 113 = 365)
N_DAYS = 365

if not PATH_INPUT.exists():
    raise FileNotFoundError(f"Input file not found: {PATH_INPUT}")

if not zipfile.is_zipfile(PATH_INPUT):
    raise ValueError(
        "Input file is not a valid .xlsx workbook (zip container). "
        f"Check file integrity: {PATH_INPUT}"
    )

print(f"Using input file: {PATH_INPUT}")
df = pd.read_excel(PATH_INPUT, engine="openpyxl")
df["report_created_at"] = pd.to_datetime(df["report_created_at"], errors="coerce")
df = df.dropna(subset=["report_created_at", "primary_diagnosis"])
df = df[(df["report_created_at"] >= DATE_START) & (df["report_created_at"] <= DATE_END)]
df["date"] = df["report_created_at"].dt.normalize()

if "user_id" in df.columns:
    daily = df.groupby(["date", "primary_diagnosis"])["user_id"].nunique().reset_index()
else:
    daily = df.groupby(["date", "primary_diagnosis"]).size().reset_index(name="n")
    daily["user_id"] = daily["n"]
daily.columns = ["date", "doenca", "casos"]

print(f"Registros diários: {len(daily)}")
print(f"Doenças: {daily['doenca'].nunique()}")


## 2. Select series with viable density

**Summary:** For each disease, computes **temporal density** (share of days with at least one case) and total cases. Keeps only series with **at least 100 days with data** and minimum total cases (>= 20) so that time series modeling is feasible.

In [ ]:
days_full = pd.date_range(start=DATE_START, end=DATE_END, freq="D")
n_days = len(days_full)

density_list = []
for doenca, grp in daily.groupby("doenca"):
    n_days_with_data = grp["date"].nunique()
    total_casos = grp["casos"].sum()
    density = n_days_with_data / n_days if n_days else 0
    density_list.append({"doenca": doenca, "density": density, "total_casos": total_casos, "n_days_with_data": n_days_with_data})

density_df = pd.DataFrame(density_list)
MIN_DAYS = 100  # pelo menos 100 dias com dados para modelar
MIN_CASOS = 20
viable = density_df[(density_df["n_days_with_data"] >= MIN_DAYS) & (density_df["total_casos"] >= MIN_CASOS)]["doenca"].tolist()
print(f"Séries viáveis (>= {MIN_DAYS} dias com dados, total_casos >= {MIN_CASOS}): {len(viable)}")
density_df.sort_values("density", ascending=False).head(15)

## 3. AutoML modeling (252 dias treino / 113 dias teste)

**Summary:** Séries viáveis têm pelo menos 100 dias com dados. As **lacunas** (dias sem casos) são preenchidas com **média móvel** (janela 7 dias, centralizada). Cada série tem 365 dias: **treino = primeiros 252 dias**, **teste = últimos 113 dias**. **FLAML** (ts_forecast) escolhe o melhor modelo; geração de forecast e métricas (RMSE, MAE, MAPE). Séries de má qualidade (ex.: MAPE alto ou IC95% cruzando zero) são sinalizadas.

In [ ]:
from flaml import AutoML
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

def safe_name(s, max_len=50):
    return s.replace("/", "-").replace("\\", "-")[:max_len]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

automl_results = []
TIME_BUDGET = 90

for i, doenca in enumerate(viable):
    print(f"  [{i+1}/{len(viable)}] {doenca}")
    sub = daily[daily["doenca"] == doenca].set_index("date")["casos"]
    sub.index = pd.DatetimeIndex(sub.index)
    series = sub.reindex(pd.DatetimeIndex(days_full)).fillna(0)
    # Preencher lacunas (dias vazios) com média móvel (janela 7 dias, centralizada)
    mask_gap = (series == 0)
    if mask_gap.any():
        rol = series.rolling(7, min_periods=1, center=True).mean()
        series = series.copy()
        series[mask_gap] = rol[mask_gap]
        # Bordas que ainda ficaram zero: preencher com ffill/bfill
        if series.eq(0).any():
            series = series.replace(0, np.nan).ffill().bfill().fillna(0)
    series = series.astype(float)
    n = len(series)
    if n < N_DAYS:
        continue
    cut = TRAIN_DAYS  # treino = primeiros 252 dias, teste = últimos 113
    train_series = series.iloc[:cut]
    test_series = series.iloc[cut:cut + TEST_DAYS]
    train_df = train_series.reset_index()
    train_df.columns = ["timestamp", "value"]
    period = len(test_series)  # 113 dias
    test_series_used = test_series

    try:
        automl = AutoML()
        automl.fit(
            dataframe=train_df,
            label="value",
            period=period,
            task="ts_forecast",
            time_budget=TIME_BUDGET,
            metric="mape",
            eval_method="holdout",
            verbose=0,
        )
        X_test = test_series_used.index.to_frame(index=False)
        X_test.columns = ["timestamp"]
        y_pred = automl.predict(X_test)
        y_pred = np.maximum(np.asarray(y_pred).flatten(), 0)
        y_test = test_series_used.values
    except Exception as e:
        print(f"    Erro: {e}")
        continue

    res_std = np.nanstd(y_test - y_pred)
    if np.isnan(res_std) or res_std <= 0:
        # Quando teste é constante (ex.: zeros), usar 0 para IC não ficar artificialmente largo
        res_std = np.nanstd(y_test) if np.isfinite(np.nanstd(y_test)) and np.nanstd(y_test) > 0 else 0.0
    lower = np.maximum(y_pred - 1.96 * res_std, 0)
    upper = y_pred + 1.96 * res_std

    best_estimator = getattr(automl, "model", None)
    try:
        cfg = getattr(automl, "best_config", None)
        model_name = cfg.get("learner") if isinstance(cfg, dict) else None
    except Exception:
        model_name = None
    if not model_name:
        est = getattr(best_estimator, "estimator", best_estimator)
        model_name = type(est).__name__ if est is not None else "AutoML"
    model_name = str(model_name)

    summary_text = ""
    try:
        est = getattr(best_estimator, "estimator", best_estimator)
        if hasattr(est, "summary"):
            s = est.summary()
            summary_text = s.as_text() if hasattr(s, "as_text") else str(s)
        else:
            summary_text = f"Model: {model_name}"
    except Exception:
        summary_text = f"Model: {model_name}"

    rmse_val = rmse(y_test, y_pred)
    mape_val = mape(y_test, y_pred)
    mae_val = mean_absolute_error(y_test, y_pred)
    # Só penalizar "IC passando pelo zero" quando o período de teste tem variância real (evita marcar como ruim séries com teste todo zero)
    test_has_variance = np.isfinite(np.nanstd(y_test)) and np.nanstd(y_test) > 1e-6
    ci_crosses_zero = np.any((y_pred - 1.96 * res_std) <= 0) and test_has_variance
    bad_metrics = (np.isfinite(mape_val) and mape_val > 100) or (test_has_variance and np.isfinite(rmse_val) and rmse_val > 2 * np.nanmean(y_test))
    if ci_crosses_zero or bad_metrics:
        obs = "Série muito ruim: "
        parts = []
        if bad_metrics:
            parts.append("métricas de desempenho elevadas (MAPE>100% ou RMSE muito alto)")
        if ci_crosses_zero:
            parts.append("intervalo de confiança 95% passando pelo zero")
        observacoes = obs + "; ".join(parts) + "."
    else:
        observacoes = ""

    automl_results.append({
        "doenca": doenca,
        "modelo": model_name,
        "y_test": y_test,
        "y_pred": y_pred,
        "lower_95": lower,
        "upper_95": upper,
        "test_index": test_series_used.index,
        "summary_text": summary_text,
        "RMSE": rmse_val,
        "MAE": mae_val,
        "MAPE": mape_val,
        "observações": observacoes,
    })

print(f"Modeladas com sucesso: {len(automl_results)} séries.")

## 4. Save model summaries

**Summary:** Writes one Excel file per series with the chosen model’s summary (e.g. statsmodels output or model name). Path: `data/results/time_series_outputs/summaries/{doenca}_{modelo}_summary.xlsx`.

In [ ]:
for r in automl_results:
    doenca = r["doenca"]
    modelo = r["modelo"]
    summary_text = r["summary_text"]
    fname = f"{safe_name(doenca)}_{safe_name(modelo)}_summary.xlsx"
    path = PATH_SUMMARIES / fname
    pd.DataFrame({"summary": [summary_text]}).to_excel(path, index=False, engine="openpyxl")
print(f"Salvos {len(automl_results)} summaries em {PATH_SUMMARIES}")

## 5. Save charts (test series + forecast + 95% CI)

**Summary:** For each series, saves one chart showing only the **test period**: observed values, point forecast, and 95% confidence band. Path: `data/results/time_series_outputs/charts/{doenca}_{modelo}_chart.png`.

In [ ]:
for r in automl_results:
    doenca = r["doenca"]
    modelo = r["modelo"]
    idx = r["test_index"]
    x_ax = idx.to_timestamp() if hasattr(idx, "to_timestamp") else idx
    y_test = r["y_test"]
    y_pred = r["y_pred"]
    lower = r["lower_95"]
    upper = r["upper_95"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_ax, y_test, label="Teste (observado)", color="C0")
    ax.plot(x_ax, y_pred, label="Forecast", color="C1")
    ax.fill_between(x_ax, lower, upper, alpha=0.3, color="C1")
    ax.set_title(f"{doenca} — {modelo}")
    ax.legend(loc="best", fontsize=8)
    ax.set_ylabel("Casos")
    plt.tight_layout()
    fname = f"{safe_name(doenca)}_{safe_name(modelo)}_chart.png"
    fig.savefig(PATH_CHARTS / fname, dpi=120, bbox_inches="tight")
    plt.close(fig)
print(f"Salvos {len(automl_results)} gráficos em {PATH_CHARTS}")

## 6. Save evaluation metrics

**Summary:** Exports a single Excel file with one row per series: disease, chosen model, RMSE, MAE, MAPE, and observations (e.g. notes when the series is poor quality). Path: `data/results/time_series_outputs/evaluation_metrics_all_models.xlsx`.

In [ ]:
metrics_df = pd.DataFrame([
    {"doenca": r["doenca"], "modelo_escolhido": r["modelo"], "RMSE": r["RMSE"], "MAE": r["MAE"], "MAPE": r["MAPE"], "observações": r.get("observações", "")}
    for r in automl_results
])
path_metrics = PATH_OUTPUT / "evaluation_metrics_all_models.xlsx"
metrics_df.to_excel(path_metrics, index=False, engine="openpyxl")
print(f"Salvo: {path_metrics}")
metrics_df.head(20)

## 7. Viable series selection summary

**Summary:** Exports an Excel file with (1) total diseases, count and list of excluded series, count and list of modeled series; (2) a per-disease table with days with cases and days filled with moving averages. Path: `data/results/time_series_outputs/evaluation_viable_series_summary.xlsx`.

In [ ]:
total_diseases = len(density_df)
excluded = density_df[~density_df["doenca"].isin(viable)]["doenca"].tolist()
modeled = [r["doenca"] for r in automl_results]
n_excluded = len(excluded)
n_modeled = len(modeled)

print("=== Resumo da seleção de séries viáveis ===")
print(f"Total de doenças: {total_diseases}")
print(f"Séries excluídas: {n_excluded}")
for d in excluded:
    print(f"  - {d}")
print(f"Séries modeladas: {n_modeled}")
for d in modeled:
    print(f"  - {d}")

n_days = len(days_full)
per_disease = density_df.copy()
per_disease["n_days_with_cases"] = per_disease["n_days_with_data"]
per_disease["n_days_filled_ma"] = per_disease.apply(
    lambda r: (n_days - r["n_days_with_data"]) if r["doenca"] in viable else 0, axis=1
)
per_disease["total_days"] = n_days
per_disease["status"] = per_disease["doenca"].apply(
    lambda d: "modeled" if d in modeled else ("viable" if d in viable else "excluded")
)
per_disease = per_disease[["doenca", "n_days_with_cases", "n_days_filled_ma", "total_days", "status", "total_casos", "density"]]

path_summary = PATH_OUTPUT / "evaluation_viable_series_summary.xlsx"
with pd.ExcelWriter(path_summary, engine="openpyxl") as writer:
    summary_df = pd.DataFrame({
        "metric": ["total_diseases", "excluded_count", "modeled_count"],
        "value": [total_diseases, n_excluded, n_modeled],
    })
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    pd.DataFrame({"excluded_series": excluded}).to_excel(writer, sheet_name="Excluded_series", index=False)
    pd.DataFrame({"modeled_series": modeled}).to_excel(writer, sheet_name="Modeled_series", index=False)
    per_disease.to_excel(writer, sheet_name="Per_disease", index=False)
print(f"Salvo: {path_summary}")
per_disease.head(15)

## 8. Outbreak detector (PoC: dengue, influenza, and SARS)

**Goal:** add a simple and explainable outbreak detection strategy at the end of the pipeline while keeping the current notebook template.

**Evaluation strategy (step by step):**
1. Select the 3 infectious diseases in this PoC using keyword matching in symptom/diagnosis labels (dengue, influenza/flu, and SARS).
2. Reindex each disease daily series to the full 365-day period and split into **train (first 252 days)** and **test (last 113 days)**.
3. Smooth daily noise with a **7-day moving average**.
4. Aggregate to weekly values and, for each week, compute a baseline from the **previous 4 weeks** (derived from smoothed daily data):
   - expected mean,
   - 95% confidence interval (95% CI),
   - upper 95th percentile (P95).
5. Flag **spike/outbreak points** where the weekly observed value is **above P95**.

This approach is useful for surveillance because it compares current activity against recent expected behavior, reduces false alarms from day-to-day noise, and highlights abrupt increases with an objective rule that is easy to communicate.

In [ ]:
# Outbreak detection PoC focused on infectious diseases
# (dengue, influenza, and SARS) with weekly spike flags above P95.

from pathlib import Path

PATH_OUTBREAK_CHARTS = PATH_OUTPUT / "outbreak_detection_charts"
os.makedirs(PATH_OUTBREAK_CHARTS, exist_ok=True)

def safe_name(text: str) -> str:
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in str(text)).strip("_")

# 1) Select target diseases by keyword matching in diagnosis labels.
keyword_groups = {
    "dengue": ["dengue"],
    "influenza": ["influenza", "gripe", "flu"],
    "sars": ["sars", "covid"],
}

all_diseases = sorted(daily["doenca"].dropna().unique().tolist())
selected_diseases = {}
for group_name, keywords in keyword_groups.items():
    matches = [d for d in all_diseases if any(k in d.lower() for k in keywords)]
    if matches:
        selected_diseases[group_name] = matches[0]

print("Selected diseases for outbreak PoC:")
for k, v in selected_diseases.items():
    print(f"  - {k}: {v}")

missing_groups = [g for g in keyword_groups if g not in selected_diseases]
if missing_groups:
    print(f"Warning: no label found for {missing_groups}. Only available matches will be modeled.")

# 2) Build outbreak detector with train/test split, 7-day MA, and weekly 4-week baseline.
test_start = pd.Timestamp(days_full[TRAIN_DAYS])
outbreak_results = []

for group_name, disease_label in selected_diseases.items():
    series = (
        daily[daily["doenca"] == disease_label]
        .set_index("date")["casos"]
        .reindex(pd.DatetimeIndex(days_full))
        .fillna(0.0)
        .astype(float)
    )

    # Split (kept for methodological consistency and reporting)
    train_daily = series.iloc[:TRAIN_DAYS]
    test_daily = series.iloc[TRAIN_DAYS:TRAIN_DAYS + TEST_DAYS]

    # 3) Smooth with 7-day moving average on daily data.
    smoothed_daily = series.rolling(window=7, min_periods=1).mean()

    # 4) Weekly signal + baseline from previous 4 weeks
    weekly = smoothed_daily.resample("W-SUN").mean().to_frame("observed_weekly")
    hist = weekly["observed_weekly"].shift(1)

    weekly["baseline_mean"] = hist.rolling(window=4, min_periods=4).mean()
    weekly["baseline_std"] = hist.rolling(window=4, min_periods=4).std(ddof=1)
    weekly["p95"] = hist.rolling(window=4, min_periods=4).quantile(0.95)

    z = 1.96
    weekly["ci_low"] = weekly["baseline_mean"] - z * weekly["baseline_std"]
    weekly["ci_high"] = weekly["baseline_mean"] + z * weekly["baseline_std"]

    # 5) Outbreak/spike if observed is above upper 95th percentile
    weekly["is_spike"] = weekly["observed_weekly"] > weekly["p95"]
    weekly["period"] = np.where(weekly.index < test_start, "train", "test")
    weekly["excess_over_p95"] = weekly["observed_weekly"] - weekly["p95"]

    # Pick spike examples (prefer test period for presentation relevance)
    valid_spikes = weekly[weekly["is_spike"] & weekly["p95"].notna()].copy()
    spike_examples = valid_spikes[valid_spikes["period"] == "test"].nlargest(3, "excess_over_p95")
    if spike_examples.empty:
        spike_examples = valid_spikes.nlargest(3, "excess_over_p95")

    # Chart with selected spikes highlighted
    fig, ax = plt.subplots(figsize=(12, 5))
    x_ax = weekly.index.to_pydatetime()

    ax.plot(x_ax, weekly["observed_weekly"], color="C0", lw=2, label="Observed weekly (7-day MA)")
    ax.plot(x_ax, weekly["baseline_mean"], color="C2", lw=1.8, ls="--", label="Expected mean (previous 4 weeks)")
    ax.plot(x_ax, weekly["p95"], color="C3", lw=1.8, ls=":", label="Upper P95 threshold")

    ci_mask = weekly["ci_low"].notna() & weekly["ci_high"].notna()
    if ci_mask.any():
        ax.fill_between(
            np.array(x_ax)[ci_mask.values],
            weekly.loc[ci_mask, "ci_low"].values,
            weekly.loc[ci_mask, "ci_high"].values,
            color="C2",
            alpha=0.15,
            label="95% CI (baseline)",
        )

    if not spike_examples.empty:
        spike_x = spike_examples.index.to_pydatetime()
        spike_y = spike_examples["observed_weekly"].values
        ax.scatter(spike_x, spike_y, s=80, color="red", zorder=5, label="Selected spike examples")
        for d, y in zip(spike_examples.index, spike_y):
            ax.annotate(d.strftime("%Y-%m-%d"), (d.to_pydatetime(), y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)

    ax.axvline(test_start.to_pydatetime(), color="gray", lw=1.2, ls="--", label="Train/Test split")
    ax.set_title(f"Outbreak detector - {group_name.title()} ({disease_label})")
    ax.set_xlabel("Week")
    ax.set_ylabel("Smoothed weekly cases")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=9)

    chart_path = PATH_OUTBREAK_CHARTS / f"{safe_name(group_name)}_{safe_name(disease_label)}_outbreak_spikes.png"
    fig.tight_layout()
    fig.savefig(chart_path, dpi=150)
    plt.close(fig)

    outbreak_results.append({
        "group": group_name,
        "disease_label": disease_label,
        "train_days": len(train_daily),
        "test_days": len(test_daily),
        "weekly_points": int(len(weekly)),
        "spikes_train": int(weekly[(weekly["period"] == "train") & (weekly["is_spike"])].shape[0]),
        "spikes_test": int(weekly[(weekly["period"] == "test") & (weekly["is_spike"])].shape[0]),
        "selected_spike_dates": ", ".join([d.strftime("%Y-%m-%d") for d in spike_examples.index]) if not spike_examples.empty else "",
        "chart_path": str(chart_path),
    })

summary_outbreak = pd.DataFrame(outbreak_results)
path_outbreak_summary = PATH_OUTPUT / "outbreak_detector_poc_summary.xlsx"
summary_outbreak.to_excel(path_outbreak_summary, index=False, engine="openpyxl")

print(f"Saved outbreak summary: {path_outbreak_summary}")
print(f"Saved charts folder: {PATH_OUTBREAK_CHARTS}")
summary_outbreak